# VoxCPM2 模型在 Kaggle GPU V100 上生成音频测试

## 环境准备
1. 开启 GPU 加速器 (Settings -> GPU -> V100)
2. 运行所有单元格

## 环境准备

In [ ]:
!pip install -q torch torchaudio transformers accelerate huggingface_hub -q 2>&1 | tail -5

In [ ]:
import torch
import torch.nn as nn
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"cuDNN: {torch.backends.cudnn.version()}")
else:
    print("CUDA not available - 请在 Kaggle 设置中开启 GPU (V100)")

In [ ]:
# 安装 voxcpm2 依赖
!pip install -q transformers accelerate safetensors huggingface_hub -q 2>&1 | tail -3

In [ ]:
# 下载 VoxCPM2 模型 (使用 Hugging Face)
from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id="tencent/VoxCPM2",
    local_dir="/kaggle/working/VoxCPM2",
    local_dir_use_symlinks=False
)
print(f"模型下载完成: {model_path}")

In [ ]:
# 检查模型结构
import os
model_path = "/kaggle/working/VoxCPM2"
for root, dirs, files in os.walk(model_path):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# VoxCPM2 推理测试
import torch
import torch.nn as nn

# 尝试加载模型
from transformers import AutoModel, AutoTokenizer

model_path = "/kaggle/working/VoxCPM2"

# 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/working/VoxCPM2",
    trust_remote_code=True
)

# 加载模型 (fp16 以节省显存)
model = AutoModel.from_pretrained(
    "/kaggle/working/VoxCPM2",
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

model.eval()
print(f"模型加载完成，设备: {next(model.parameters()).device}")
print(f"模型参数量: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

In [ ]:
# 测试生成音频
import torch
import torchaudio
import tempfile
import soundfile as sf

# 测试文本
texts = [
    "Hello, this is a test of VoxCPM2 text to speech synthesis.",
    "你好，这是 VoxCPM2 模型生成的中文语音测试。",
    "The quick brown fox jumps over the lazy dog."
]

# VoxCPM2 推理示例
@torch.no_grad()
def generate_audio(text, speaker_id=0, temperature=0.7):
    """生成音频"""
    # 编码文本
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    # 生成语音码
    with torch.no_grad():
        # VoxCPM2 使用流式生成
        # 具体 API 取决于模型实现
        pass
    
    # 这里需要根据实际的 VoxCPM2 API 调整
    # 通常包含：encode text -> generate codes -> decode to audio
    return None

# 测试生成
text = texts[0]
print(f"Testing: {text}")
# audio = generate_audio(text)
# print(f"Generated audio shape: {audio.shape if audio is not None else 'None'}")

print("测试代码准备完成，等待模型加载...")

In [ ]:
# 完整的推理流程封装
import torch
import torchaudio
import tempfile
import os

class VoxCPM2Inference:
    def __init__(self, model_path="/kaggle/working/VoxCPM2"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model_path = model_path
        self.model = None
        self.tokenizer = None
        self._load_model()
    
    def _load_model(self):
        from transformers import AutoModel, AutoTokenizer
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_path,
            trust_remote_code=True
        )
        
        from transformers import AutoModel
        self.model = AutoModel.from_pretrained(
            self.model_path,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
            low_cpu_mem_usage=True
        )
        self.model.eval()
        
    def synthesize(self, text: str, speaker_id: int = 0, temperature: float = 0.7) -> bytes:
        """合成语音，返回 WAV 字节"""
        # 这里需要根据 VoxCPM2 实际 API 实现
        # 参考: https://github.com/TencentGameMate/VoxCPM2
        pass
    
    def save_audio(self, audio_tensor, sample_rate: int, output_path: str):
        """保存音频文件"""
        torchaudio.save(output_path, audio_tensor.cpu(), 24000)
        return output_path

# 初始化
infer = VoxCPM2Inference()

# 测试文本
texts = [
    "Hello, this is a test of VoxCPM2 text to speech synthesis.",
    "你好，这是 VoxCPM2 模型生成的中文语音测试。",
]

for i, text in enumerate(test_texts):
    try:
        audio = infer.synthesize(text, speaker_id=0)
        if audio is not None:
            output = f"/kaggle/working/output_{i}.wav"
            infer.save_audio(audio, 24000, output)
            print(f"✅ 生成成功: {output}")
        else:
            print(f"⚠️  {text[:30]}... - 推理 API 需根据实际模型调整")
    except Exception as e:
        print(f"❌ 错误: {e}")

In [ ]:
# 验证生成的音频文件
import os
for f in os.listdir("/kaggle/working"):
    if f.endswith(".wav"):
        path = f"/kaggle/working/{f}"
        import soundfile as sf
        info = sf.info(path)
        print(f"{f}: {info.duration:.2f}s, {info.samplerate}Hz, {info.channels}ch, {info.format}")

## 结果分析

1. **模型加载**: 检查模型是否正确加载到 GPU
2. **推理速度**: 观察 RTF (Real-Time Factor = 合成时间 / 音频时长)
2. **显存占用**: 观察 GPU 显存占用
3. **音频质量**: 听取生成的音频质量

## 预期指标 (VoxCPM2 V100 FP16):
- RTF (实时率): ~0.05-0.1 (即 1秒音频需 0.05-0.1 秒合成)
- 显存占用: ~10-12 GB (FP16) / ~8-10 GB (INT8)
- 音频采样率: 24kHz 或 48kHz
- 支持: 零样本音色克隆、语速控制、情感控制